## Example 1: Basic Reactive Descriptors

In [ ]:
# Example 1: Basic reactive descriptors and attribute watching
print("Example 1: Basic reactive descriptors and attribute watching")

try:
    from reacttrs import Int, Str, Bool, Float, watch
    
    from importlib.metadata import version as _distribution_version
    print(f"ReactTRS version: {_distribution_version('reacttrs')}")
    print("ReactTRS provides reactive descriptors for Python classes")
    
    # Create a reactive model class
    class UserProfile:
        user_id = Int(0)
        username = Str("anonymous")
        is_active = Bool(False)
        balance = Float(0.0)
        
        def __init__(self, user_id=None, username=None):
            if user_id is not None:
                self.user_id = user_id
            if username is not None:
                self.username = username
    
    # Create user instances
    user1 = UserProfile(1, "alice")
    user2 = UserProfile(2, "bob")
    
    print(f"\nInitial user profiles:")
    print(f"User 1: ID={user1.user_id}, Name={user1.username}, Active={user1.is_active}, Balance=${user1.balance}")
    print(f"User 2: ID={user2.user_id}, Name={user2.username}, Active={user2.is_active}, Balance=${user2.balance}")
    
    # Modify attributes (reactive descriptors work transparently)
    user1.is_active = True
    user1.balance = 150.75
    user2.is_active = True
    user2.balance = 200.50
    
    print(f"\nAfter activation and balance updates:")
    print(f"User 1: ID={user1.user_id}, Name={user1.username}, Active={user1.is_active}, Balance=${user1.balance}")
    print(f"User 2: ID={user2.user_id}, Name={user2.username}, Active={user2.is_active}, Balance=${user2.balance}")
    
    # Asserted BEFORE the type-safety block below reassigns user1.balance to 99.99.
    assert (user1.balance, user2.balance) == (150.75, 200.50), (user1.balance, user2.balance)
    assert user1.is_active and user2.is_active, (user1.is_active, user2.is_active)

    # Test different instances maintain separate state
    user1.username = "alice_updated"
    user2.username = "bob_premium"
    
    print(f"\nAfter username updates (showing instance isolation):")
    print(f"User 1: {user1.username}")
    print(f"User 2: {user2.username}")
    assert (user1.username, user2.username) == ("alice_updated", "bob_premium"), (user1.username, user2.username)
    
    # Demonstrate type safety
    print(f"\nTesting type safety:")
    try:
        user1.user_id = 42
        print(f"  Valid int assignment: user_id = {user1.user_id}")
        user1.balance = 99.99
        print(f"  Valid float assignment: balance = {user1.balance}")
        user1.is_active = False
        print(f"  Valid bool assignment: is_active = {user1.is_active}")
        assert (user1.user_id, user1.balance, user1.is_active) == (42, 99.99, False), user1.user_id
    except Exception as type_error:
        print(f"  Type error: {type_error}")
        raise
    
    # Show descriptor behavior
    print(f"\nDescriptor information:")
    print(f"  UserProfile.user_id is descriptor: {hasattr(UserProfile.user_id, '__get__')}")
    print(f"  UserProfile.username is descriptor: {hasattr(UserProfile.username, '__get__')}")
    
    print("Basic reactive descriptors demonstration completed successfully")
    
except Exception as e:
    print(f"Error in reactive descriptors: {e}")
    raise AssertionError(
        f"reacttrs example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Watcher Functions and Change Tracking

In [ ]:
# Example 2: Watcher functions and attribute change tracking
print("Example 2: Watcher functions and attribute change tracking")

try:
    from reacttrs import Int, Str, Bool, Float, watch
    
    # Create a model with reactive attributes for monitoring
    class ServerConfig:
        port = Int(8080)
        hostname = Str("localhost")
        debug_mode = Bool(False)
        max_connections = Int(100)
        timeout = Float(30.0)
        
        def __init__(self):
            self.change_log = []
        
        def log_change(self, attribute, old_value, new_value):
            self.change_log.append({
                'attribute': attribute,
                'old_value': old_value,
                'new_value': new_value,
                'timestamp': len(self.change_log) + 1
            })
    
    # Create server configuration instance
    config = ServerConfig()
    
    print("=== Setting up watchers ===")
    
    # Define watcher functions for different attributes
    def port_watcher(obj, old_value, new_value):
        obj.log_change('port', old_value, new_value)
        print(f"  🔧 Port changed: {old_value} → {new_value}")
        if new_value < 1024:
            print(f"    ⚠️  Warning: Port {new_value} requires root privileges")
    
    def hostname_watcher(obj, old_value, new_value):
        obj.log_change('hostname', old_value, new_value)
        print(f"  🌐 Hostname changed: '{old_value}' → '{new_value}'")
        if new_value == "0.0.0.0":
            print(f"    ⚠️  Warning: Binding to all interfaces")
    
    def debug_watcher(obj, old_value, new_value):
        obj.log_change('debug_mode', old_value, new_value)
        print(f"  🐛 Debug mode changed: {old_value} → {new_value}")
        if new_value:
            print(f"    ℹ️  Debug logging enabled")
        else:
            print(f"    ℹ️  Debug logging disabled")
    
    def connections_watcher(obj, old_value, new_value):
        obj.log_change('max_connections', old_value, new_value)
        print(f"  👥 Max connections changed: {old_value} → {new_value}")
        if new_value > 1000:
            print(f"    ⚠️  Warning: High connection limit may impact performance")
    
    def timeout_watcher(obj, old_value, new_value):
        obj.log_change('timeout', old_value, new_value)
        print(f"  ⏰ Timeout changed: {old_value}s → {new_value}s")
    
    # Register watchers for each attribute
    watch(config, 'port', port_watcher)
    watch(config, 'hostname', hostname_watcher)
    watch(config, 'debug_mode', debug_watcher)
    watch(config, 'max_connections', connections_watcher)
    watch(config, 'timeout', timeout_watcher)
    
    print(f"Initial configuration:")
    print(f"  Port: {config.port}")
    print(f"  Hostname: {config.hostname}")
    print(f"  Debug: {config.debug_mode}")
    print(f"  Max connections: {config.max_connections}")
    print(f"  Timeout: {config.timeout}s")
    
    print(f"\n=== Making configuration changes ===")
    
    # Make changes that trigger watchers
    config.port = 3000
    config.hostname = "0.0.0.0"
    config.debug_mode = True
    config.max_connections = 1500
    config.timeout = 60.0
    
    print(f"\n=== Additional changes ===")
    config.port = 443  # This should trigger root privilege warning
    config.debug_mode = False
    config.max_connections = 50
    
    print(f"\n=== Change log analysis ===")
    print(f"Total changes recorded: {len(config.change_log)}")
    for i, change in enumerate(config.change_log, 1):
        print(f"  {i}. {change['attribute']}: {change['old_value']} → {change['new_value']}")
    
    # Eight assignments, each one a change, so eight watcher calls -- asserted BEFORE the secondary watcher below
    # sets port 8443, after which the log holds nine.
    assert [change["attribute"] for change in config.change_log] == [
        "port", "hostname", "debug_mode", "max_connections", "timeout", "port", "debug_mode", "max_connections"
    ], config.change_log
    assert config.change_log[5]["old_value"] == 3000 and config.change_log[5]["new_value"] == 443, config.change_log[5]

    # Demonstrate multiple watchers on same attribute
    print(f"\n=== Multiple watchers on same attribute ===")
    
    def secondary_port_watcher(obj, old_value, new_value):
        print(f"  📊 Secondary watcher: Port statistics updated")
    
    watch(config, 'port', secondary_port_watcher)
    
    print(f"Adding second watcher for port changes:")
    config.port = 8443  # This should trigger both watchers
    
    print(f"\n=== Final configuration state ===")
    print(f"  Port: {config.port}")
    print(f"  Hostname: {config.hostname}")
    print(f"  Debug: {config.debug_mode}")
    print(f"  Max connections: {config.max_connections}")
    print(f"  Timeout: {config.timeout}s")
    print(f"  Total changes: {len(config.change_log)}")
    # The secondary watcher does not log, so the 8443 change adds exactly one entry (from port_watcher).
    assert len(config.change_log) == 9 and config.port == 8443, (len(config.change_log), config.port)
    assert config.change_log[-1]["old_value"] == 443 and config.change_log[-1]["new_value"] == 8443, config.change_log[-1]
    
    print("Watcher functions and change tracking completed successfully")
    
except Exception as e:
    print(f"Error in watcher functions: {e}")
    raise AssertionError(
        f"reacttrs example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Validation and Reactive Data Models

In [ ]:
# Example 3: Validation and reactive data models
print("Example 3: Validation and reactive data models")

try:
    from reacttrs import Int, Str, Bool, Float, watch, Reactive
    
    # Create a comprehensive data model with validation
    class Product:
        # Basic reactive attributes with validation
        product_id = Int(0)
        name = Str("")
        price = Float(0.0)
        in_stock = Bool(True)
        quantity = Int(0)
        category = Str("general")
        
        def __init__(self, product_id, name, price, quantity=0):
            self.product_id = product_id
            self.name = name
            self.price = price
            self.quantity = quantity
            self.validation_errors = []
            self._setup_watchers()
        
        def _setup_watchers(self):
            """Set up validation watchers for the product"""
            watch(self, 'price', self._validate_price)
            watch(self, 'quantity', self._validate_quantity)
            watch(self, 'name', self._validate_name)
            watch(self, 'product_id', self._validate_id)
        
        def _validate_price(self, obj, old_value, new_value):
            if new_value < 0:
                error = f"Price cannot be negative: {new_value}"
                self.validation_errors.append(error)
                print(f"  ❌ Validation Error: {error}")
                # Reset to previous valid value
                self.price = old_value
            elif new_value > 10000:
                print(f"  ⚠️  Warning: High price {new_value} - please confirm")
            else:
                print(f"  ✅ Price validation passed: ${new_value:.2f}")
        
        def _validate_quantity(self, obj, old_value, new_value):
            if new_value < 0:
                error = f"Quantity cannot be negative: {new_value}"
                self.validation_errors.append(error)
                print(f"  ❌ Validation Error: {error}")
                self.quantity = old_value
            else:
                # Update stock status based on quantity
                self.in_stock = new_value > 0
                print(f"  ✅ Quantity validation passed: {new_value} (In stock: {self.in_stock})")
        
        def _validate_name(self, obj, old_value, new_value):
            if len(new_value.strip()) < 2:
                error = f"Product name too short: '{new_value}'"
                self.validation_errors.append(error)
                print(f"  ❌ Validation Error: {error}")
                self.name = old_value
            else:
                print(f"  ✅ Name validation passed: '{new_value}'")
        
        def _validate_id(self, obj, old_value, new_value):
            if new_value <= 0:
                error = f"Product ID must be positive: {new_value}"
                self.validation_errors.append(error)
                print(f"  ❌ Validation Error: {error}")
                self.product_id = old_value
            else:
                print(f"  ✅ ID validation passed: {new_value}")
        
        def __str__(self):
            stock_status = "In Stock" if self.in_stock else "Out of Stock"
            return f"Product({self.product_id}): {self.name} - ${self.price:.2f} ({self.quantity} units, {stock_status})"
    
    print("=== Creating products with validation ===")
    
    # Create product instances
    product1 = Product(1, "Laptop", 999.99, 5)
    product2 = Product(2, "Mouse", 29.99, 50)
    
    print(f"Initial products:")
    print(f"  {product1}")
    print(f"  {product2}")
    
    print(f"\n=== Testing valid changes ===")
    
    # Valid changes
    product1.price = 899.99  # Price reduction
    product1.quantity = 3    # Quantity update
    product2.name = "Wireless Gaming Mouse"  # Name update
    
    print(f"\nAfter valid changes:")
    print(f"  {product1}")
    print(f"  {product2}")
    
    print(f"\n=== Testing validation failures ===")
    
    # Test validation failures
    print("Attempting invalid changes:")
    
    # Try negative price (should fail)
    print("1. Setting negative price:")
    product1.price = -50.0
    
    # Try negative quantity (should fail)
    print("2. Setting negative quantity:")
    product1.quantity = -10
    
    # Try short name (should fail)
    print("3. Setting short name:")
    product2.name = "X"
    
    # Try zero/negative ID (should fail)
    print("4. Setting invalid ID:")
    product1.product_id = -1
    
    print(f"\nAfter validation failures (values should be unchanged):")
    print(f"  {product1}")
    print(f"  {product2}")
    
    # Each validation watcher reverted its invalid value, and recorded why.
    assert (product1.price, product1.quantity, product1.product_id) == (899.99, 3, 1), (
        product1.price, product1.quantity, product1.product_id
    )
    assert product2.name == "Wireless Gaming Mouse", product2.name
    assert len(product1.validation_errors) == 3 and len(product2.validation_errors) == 1, (
        product1.validation_errors, product2.validation_errors
    )

    # Test warnings
    print(f"\n=== Testing validation warnings ===")
    print("Setting high price (should warn but accept):")
    product1.price = 15000.0
    
    print(f"\n=== Inventory management simulation ===")
    
    # Create inventory system
    class InventoryManager:
        def __init__(self):
            self.products = []
            self.total_value = Float(0.0)
            self.total_items = Int(0)
        
        def add_product(self, product):
            self.products.append(product)
            # Watch for changes to update totals
            watch(product, 'quantity', self._update_totals)
            watch(product, 'price', self._update_totals)
            self._update_totals(None, None, None)  # Initial calculation
        
        def _update_totals(self, obj, old_value, new_value):
            self.total_value = sum(p.price * p.quantity for p in self.products)
            self.total_items = sum(p.quantity for p in self.products)
            print(f"  📊 Inventory updated: {self.total_items} items, ${self.total_value:.2f} total value")
    
    inventory = InventoryManager()
    inventory.add_product(product1)
    inventory.add_product(product2)
    
    print(f"\nMaking inventory changes:")
    product1.quantity = 10  # Should trigger inventory update
    product2.quantity = 0   # Should mark as out of stock and update inventory
    
    # The quantity watcher keeps in_stock in step, and the inventory watchers keep the totals in step.
    assert (product2.quantity, product2.in_stock) == (0, False), (product2.quantity, product2.in_stock)
    assert (inventory.total_items, inventory.total_value) == (10, 150000.0), (inventory.total_items, inventory.total_value)

    print(f"\nFinal validation error summary:")
    all_errors = product1.validation_errors + product2.validation_errors
    if all_errors:
        for i, error in enumerate(all_errors, 1):
            print(f"  {i}. {error}")
    else:
        print("  No validation errors recorded")
    
    print("Validation and reactive data models completed successfully")
    
except Exception as e:
    print(f"Error in validation and data models: {e}")
    raise AssertionError(
        f"reacttrs example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Advanced Reactive Patterns

In [ ]:
# Example 4: Advanced reactive patterns and state management
print("Example 4: Advanced reactive patterns and state management")

try:
    from reacttrs import Int, Str, Bool, Float, watch, Reactive
    
    # Create a complex reactive state management system
    class GameState:
        # Player state
        player_health = Int(100)
        player_mana = Int(50)
        player_level = Int(1)
        player_experience = Int(0)
        player_name = Str("Hero")
        
        # Game state
        current_room = Str("town")
        game_paused = Bool(False)
        score = Int(0)
        time_elapsed = Float(0.0)
        
        # Combat state
        in_combat = Bool(False)
        enemy_health = Int(0)
        combat_round = Int(0)
        
        def __init__(self, player_name="Hero"):
            self.player_name = player_name
            self.event_log = []
            self.state_history = []
            self._setup_reactive_system()
        
        def _setup_reactive_system(self):
            """Set up complex reactive relationships"""
            # Health monitoring
            watch(self, 'player_health', self._on_health_change)
            watch(self, 'player_experience', self._check_level_up)
            watch(self, 'current_room', self._on_room_change)
            watch(self, 'in_combat', self._on_combat_state_change)
            watch(self, 'enemy_health', self._check_combat_end)
        
        def _log_event(self, event):
            self.event_log.append(f"[{len(self.event_log)+1}] {event}")
            print(f"  📝 {event}")
        
        def _on_health_change(self, obj, old_value, new_value):
            if new_value <= 0:
                self._log_event(f"💀 {self.player_name} has died!")
                self.game_paused = True
                self.in_combat = False
            elif new_value < old_value:
                damage = old_value - new_value
                self._log_event(f"💥 {self.player_name} takes {damage} damage (Health: {new_value}/100)")
            elif new_value > old_value:
                healing = new_value - old_value
                self._log_event(f"💚 {self.player_name} heals {healing} HP (Health: {new_value}/100)")
        
        def _check_level_up(self, obj, old_value, new_value):
            xp_for_next_level = self.player_level * 100
            if new_value >= xp_for_next_level:
                old_level = self.player_level
                self.player_level += 1
                self.player_experience = new_value - xp_for_next_level
                self.player_health = min(100, self.player_health + 20)  # Level up healing
                self.player_mana = min(100, self.player_mana + 10)
                self._log_event(f"🎉 Level Up! {self.player_name} reaches level {self.player_level}")
                self.score += old_level * 50
        
        def _on_room_change(self, obj, old_value, new_value):
            self._log_event(f"🚪 {self.player_name} moves from {old_value} to {new_value}")
            # Room-specific effects
            if new_value == "dungeon":
                self._log_event("⚠️  Entering dangerous area!")
            elif new_value == "town":
                self._log_event("🏠 Safe in town - health and mana regenerating")
                self.player_health = min(100, self.player_health + 10)
                self.player_mana = min(100, self.player_mana + 15)
        
        def _on_combat_state_change(self, obj, old_value, new_value):
            if new_value and not old_value:
                self.combat_round = 1
                self._log_event("⚔️  Combat begins!")
            elif old_value and not new_value:
                self.combat_round = 0
                self.enemy_health = 0
                self._log_event("✅ Combat ended!")
        
        def _check_combat_end(self, obj, old_value, new_value):
            if self.in_combat and new_value <= 0 and old_value > 0:
                self._log_event("🏆 Enemy defeated!")
                xp_gained = 25 + (self.player_level * 5)
                self.player_experience += xp_gained
                self.score += 100
                self.in_combat = False
        
        def save_state_snapshot(self):
            """Save current state for history tracking"""
            snapshot = {
                'health': self.player_health,
                'mana': self.player_mana,
                'level': self.player_level,
                'experience': self.player_experience,
                'room': self.current_room,
                'score': self.score,
                'timestamp': len(self.state_history)
            }
            self.state_history.append(snapshot)
            return snapshot
    
    print("=== Creating reactive game state ===")
    
    # Create game instance
    game = GameState("Alice")
    initial_state = game.save_state_snapshot()
    
    print(f"Initial state: Level {game.player_level}, Health {game.player_health}, Location: {game.current_room}")
    
    print(f"\n=== Simulating gameplay ===")
    
    # Simulate game events
    print("1. Exploring and gaining experience:")
    game.player_experience = 80
    game.current_room = "forest"
    
    print("2. More experience to trigger level up:")
    game.player_experience = 120  # Should trigger level up
    game.save_state_snapshot()
    
    print("3. Entering dungeon and combat:")
    game.current_room = "dungeon"
    game.in_combat = True
    game.enemy_health = 50
    
    print("4. Combat simulation:")
    game.player_health = 85  # Take damage
    game.combat_round = 2
    game.enemy_health = 25   # Deal damage to enemy
    
    print("5. Defeating enemy:")
    game.enemy_health = 0    # Should end combat and give XP
    game.save_state_snapshot()
    
    print("6. Return to town for healing:")
    game.current_room = "town"
    
    print("7. Critical situation - low health:")
    game.player_health = 5   # Dangerous health level
    
    print("8. More experience for another level:")
    game.player_experience = 180  # Should level up again
    game.save_state_snapshot()
    
    # Advanced reactive pattern - conditional watchers
    print(f"\n=== Advanced Pattern: Conditional Reactions ===")
    
    def critical_health_watcher(obj, old_value, new_value):
        if new_value <= 20 and obj.current_room != "town":
            obj._log_event(f"🚨 CRITICAL: {obj.player_name} should retreat to town!")
    
    def mana_efficiency_watcher(obj, old_value, new_value):
        if new_value < 10:
            obj._log_event(f"🔵 Low mana warning: Consider resting")
    
    # Add conditional watchers
    watch(game, 'player_health', critical_health_watcher)
    watch(game, 'player_mana', mana_efficiency_watcher)
    
    print("Testing conditional watchers:")
    game.current_room = "dungeon"  # Dangerous location
    game.player_health = 15        # Critical health in dangerous location
    game.player_mana = 8          # Low mana
    
    print(f"\n=== State Analysis ===")
    print(f"Final state: Level {game.player_level}, Health {game.player_health}/{100}")
    print(f"Experience: {game.player_experience}, Score: {game.score}")
    print(f"Location: {game.current_room}, Combat: {game.in_combat}")
    
    print(f"\nEvent log ({len(game.event_log)} events):")
    for event in game.event_log[-8:]:  # Show last 8 events
        print(f"  {event}")
    
    print(f"\nState history ({len(game.state_history)} snapshots):")
    for i, snapshot in enumerate(game.state_history):
        print(f"  {i+1}. Level {snapshot['level']}, Health {snapshot['health']}, Score {snapshot['score']}, Room: {snapshot['room']}")
    
    print("Advanced reactive patterns and state management completed successfully")
    
except Exception as e:
    print(f"Error in advanced reactive patterns: {e}")
    raise AssertionError(
        f"reacttrs example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Production Reactive Architecture

In [ ]:
# Example 5: Production patterns and reactive architecture
print("Example 5: Production patterns and reactive architecture")

try:
    from reacttrs import Int, Str, Bool, Float, watch, Reactive
    import time
    from typing import List, Dict, Any
    
    # Production-ready reactive application architecture
    class ReactiveApplication:
        # Application configuration
        app_name = Str("ReactiveApp")
        version = Str("1.0.0")
        debug_mode = Bool(False)
        max_users = Int(100)
        uptime = Float(0.0)
        
        # Performance metrics
        request_count = Int(0)
        error_count = Int(0)
        response_time = Float(0.0)
        memory_usage = Float(0.0)
        
        # System state
        is_running = Bool(False)
        maintenance_mode = Bool(False)
        health_status = Str("unknown")
        
        def __init__(self, app_name="ReactiveApp"):
            self.app_name = app_name
            self.metrics_history = []
            self.alert_handlers = []
            self.performance_thresholds = {
                'max_response_time': 1000.0,  # ms
                'max_error_rate': 5.0,        # %
                'max_memory_usage': 80.0      # %
            }
            self._setup_monitoring()
        
        def _setup_monitoring(self):
            """Set up comprehensive monitoring system"""
            # Performance monitoring
            watch(self, 'response_time', self._monitor_performance)
            watch(self, 'error_count', self._track_errors)
            watch(self, 'memory_usage', self._monitor_memory)
            
            # System state monitoring
            watch(self, 'is_running', self._handle_startup_shutdown)
            watch(self, 'maintenance_mode', self._handle_maintenance)
            watch(self, 'debug_mode', self._configure_logging)
        
        def _log_metric(self, metric_type, value, threshold=None, status="OK"):
            """Log performance metrics with threshold checking"""
            metric = {
                'type': metric_type,
                'value': value,
                'threshold': threshold,
                'status': status,
                'timestamp': time.time()
            }
            self.metrics_history.append(metric)
            
            if threshold and value > threshold:
                self._trigger_alert(f"{metric_type.upper()} threshold exceeded: {value} > {threshold}")
            
            print(f"  📊 {metric_type}: {value} {f'(threshold: {threshold})' if threshold else ''} - {status}")
        
        def _trigger_alert(self, message):
            """Trigger system alerts"""
            alert = {'message': message, 'timestamp': time.time(), 'severity': 'WARNING'}
            print(f"  🚨 ALERT: {message}")
            
            for handler in self.alert_handlers:
                try:
                    handler(alert)
                except Exception as e:
                    print(f"    Alert handler failed: {e}")
        
        def _monitor_performance(self, obj, old_value, new_value):
            """Monitor application performance metrics"""
            threshold = self.performance_thresholds['max_response_time']
            status = "SLOW" if new_value > threshold else "OK"
            self._log_metric("response_time", new_value, threshold, status)
            
            # Auto-adjust based on performance
            if new_value > threshold * 2:
                if not self.maintenance_mode:
                    print(f"    🔧 Auto-enabling maintenance mode due to poor performance")
                    self.maintenance_mode = True
        
        def _track_errors(self, obj, old_value, new_value):
            """Track error rates and system health"""
            if self.request_count > 0:
                error_rate = (new_value / self.request_count) * 100
                threshold = self.performance_thresholds['max_error_rate']
                status = "UNHEALTHY" if error_rate > threshold else "HEALTHY"
                self._log_metric("error_rate", error_rate, threshold, status)
                
                # Update health status based on error rate
                if error_rate > threshold * 2:
                    self.health_status = "critical"
                elif error_rate > threshold:
                    self.health_status = "warning"
                else:
                    self.health_status = "healthy"
        
        def _monitor_memory(self, obj, old_value, new_value):
            """Monitor memory usage"""
            threshold = self.performance_thresholds['max_memory_usage']
            status = "HIGH" if new_value > threshold else "OK"
            self._log_metric("memory_usage", new_value, threshold, status)
        
        def _handle_startup_shutdown(self, obj, old_value, new_value):
            """Handle application startup and shutdown"""
            if new_value and not old_value:
                print(f"  🚀 Application '{self.app_name}' starting up...")
                self.health_status = "starting"
                self.uptime = 0.0
            elif old_value and not new_value:
                print(f"  🛑 Application '{self.app_name}' shutting down...")
                self.health_status = "stopped"
                # Log final metrics
                self._generate_session_report()
        
        def _handle_maintenance(self, obj, old_value, new_value):
            """Handle maintenance mode changes"""
            if new_value and not old_value:
                print(f"  🔧 Entering maintenance mode - limiting functionality")
                self.max_users = min(10, self.max_users)
            elif old_value and not new_value:
                print(f"  ✅ Exiting maintenance mode - restoring full functionality")
                self.max_users = 100
        
        def _configure_logging(self, obj, old_value, new_value):
            """Configure logging based on debug mode"""
            if new_value and not old_value:
                print(f"  🐛 Debug mode enabled - verbose logging activated")
            elif old_value and not new_value:
                print(f"  📝 Debug mode disabled - standard logging restored")
        
        def add_alert_handler(self, handler_func):
            """Add custom alert handlers"""
            self.alert_handlers.append(handler_func)
            print(f"  📢 Alert handler registered")
        
        def simulate_load(self, requests=100, avg_response_time=500, error_rate=2):
            """Simulate application load for testing"""
            print(f"\n  Simulating {requests} requests with {error_rate}% error rate...")
            
            self.request_count += requests
            self.error_count += int(requests * error_rate / 100)
            self.response_time = avg_response_time
            self.memory_usage = min(95, self.memory_usage + (requests / 20))
            self.uptime += requests / 100  # Simulate time passage
        
        def _generate_session_report(self):
            """Generate comprehensive session report"""
            print(f"\n  📊 Session Report for '{self.app_name}':")
            print(f"    Total requests: {self.request_count}")
            print(f"    Total errors: {self.error_count}")
            print(f"    Final health: {self.health_status}")
            print(f"    Uptime: {self.uptime:.1f}s")
            print(f"    Metrics recorded: {len(self.metrics_history)}")
        
        def get_health_check(self):
            """Get current health status"""
            return {
                'app_name': self.app_name,
                'version': self.version,
                'status': self.health_status,
                'is_running': self.is_running,
                'maintenance_mode': self.maintenance_mode,
                'uptime': self.uptime,
                'metrics': {
                    'request_count': self.request_count,
                    'error_count': self.error_count,
                    'response_time': self.response_time,
                    'memory_usage': self.memory_usage
                }
            }
    
    print("=== Production Reactive Application ===")
    
    # Create application instance
    app = ReactiveApplication("ProductionApp")
    
    # Add custom alert handler
    def email_alert_handler(alert):
        print(f"    📧 Email sent: {alert['message']}")
    
    def slack_alert_handler(alert):
        print(f"    💬 Slack notification: {alert['message']}")
    
    app.add_alert_handler(email_alert_handler)
    app.add_alert_handler(slack_alert_handler)
    
    print(f"\n=== Application Lifecycle ===")
    
    # Start application
    app.is_running = True
    app.debug_mode = True
    
    print(f"\n=== Load Testing ===")
    
    # Normal load
    app.simulate_load(50, 200, 1)  # 50 requests, 200ms avg, 1% error rate
    
    # Increased load
    app.simulate_load(100, 800, 3)  # Higher response time and error rate
    
    # High load triggering alerts
    app.simulate_load(200, 1500, 8)  # Should trigger multiple alerts
    
    # Recovery simulation
    print(f"\n=== Recovery Simulation ===")
    app.response_time = 300  # Performance improves
    app.memory_usage = 45    # Memory usage drops
    app.maintenance_mode = False  # Exit maintenance
    
    print(f"\n=== Health Check ===")
    health = app.get_health_check()
    for key, value in health.items():
        if isinstance(value, dict):
            print(f"  {key}:")
            for sub_key, sub_value in value.items():
                print(f"    {sub_key}: {sub_value}")
        else:
            print(f"  {key}: {value}")
    
    print(f"\n=== Shutdown ===")
    app.is_running = False
    
    print(f"\n=== Architecture Benefits ===")
    print("✅ Automatic monitoring and alerting")
    print("✅ Self-healing through reactive responses")
    print("✅ Comprehensive metrics tracking")
    print("✅ Configurable thresholds and handlers")
    print("✅ Production-ready health checks")
    
    print("Production patterns and reactive architecture completed successfully")
    
except Exception as e:
    print(f"Error in production patterns: {e}")
    raise AssertionError(
        f"reacttrs example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e